# Batch 3 — Pre-Scan Summary




In [1]:
import pandas as pd
from pathlib import Path
csv_path = Path(r'../data/batch3_pre_scan_summary.csv').resolve()
df = pd.read_csv(csv_path)
df

,filename,sex,reference_image,sample_area_cm2,bone_area_cm2,total_weight_g,soft_weight_g,lean_weight_g,fat_weight_g,fat_percent,BMC_g,BMD_mg_per_cm2
0,B3_M_0.txt,Male,B3_M_0,32.210,8.954,32.2112,31.5982,21.8161,9.7821,30.958,0.61298,68.459
1,B3_M_1.txt,Male,B3_M_1,33.431,9.247,33.0832,32.4282,22.4496,9.9786,30.771,0.65492,70.823
2,B3_M_2.txt,Male,B3_M_2,31.485,8.976,33.0242,32.4088,20.8957,11.5131,35.525,0.61539,68.562
3,B3_M_3.txt,Male,B3_M_3,30.260,8.223,31.0678,30.4770,20.2096,10.2674,33.689,0.59078,71.843
4,B3_M_4.txt,Male,B3_M_4,30.971,9.347,31.5254,30.8584,21.4933,9.3651,30.349,0.66701,71.363
5,B3_F_0.txt,Female,B3_F_0,27.859,8.473,24.2944,23.7460,16.6934,7.0527,29.700,0.54836,64.722
6,B3_F_1.txt,Female,B3_F_1,26.611,8.412,23.5375,22.9747,16.6927,6.2820,27.343,0.56277,66.899
7,B3_F_2.txt,Female,B3_F_2,28.908,8.786,25.2955,24.7015,17.2232,7.4784,30.275,0.59394,67.604
8,B3_F_3.txt,Female,B3_F_3,27.000,8.361,23.6029,23.0464,16.8017,6.2447,27.096,0.55647,66.557
9,B3_F_4.txt,Female,B3_F_4,27.919,8.274,24.6455,24.0841,18.9075,5.1766,21.494,0.56147,67.859


## Batch 3 — 1-week post-treatment summary


In [ ]:
# Load week-1 master CSV if present, otherwise scan for week-1 TXT files and build one; then display rows for this batch
from pathlib import Path
import re

DATA_CSV = Path(r'../data/week1_reports.csv').resolve()
batch_name = 'Batch 3'

def scan_and_build(downloads_root):
    week1_re = re.compile(r'(?:week[\s_-]*1|1[\s_-]*week|wk[\s_-]*1|week1|1week)', re.I)
    patterns = {
        'sample_area': re.compile(r"Sample Area:\s*([0-9.]+)\s*cm\^2"),
        'bone_area': re.compile(r"Bone Area:\s*([0-9.]+)\s*cm\^2"),
        'total_weight': re.compile(r"Total Weight:\s*([0-9.]+)\s*g"),
        'soft_weight': re.compile(r"Soft Weight:\s*([0-9.]+)\s*g"),
        'lean_weight': re.compile(r"Lean Weight:\s*([0-9.]+)\s*g"),
        'fat_weight': re.compile(r"Fat Weight:\s*([0-9.]+)\s*g"),
        'fat_percent': re.compile(r"Fat Percent:\s*([0-9.]+)"),
        'BMC': re.compile(r"BMC:\s*([0-9.]+)\s*g"),
        'BMD': re.compile(r"BMD:\s*([0-9.]+)\s*mg/cm\^2"),
    }
    rows = []
    for txt in downloads_root.rglob('*.txt'):
        nl = str(txt).lower()
        if week1_re.search(nl):
            text = txt.read_text(encoding='utf-8', errors='replace')
            inside_block = ''
            whole_block = ''
            if 'INSIDE ROI TISSUE STATISTICS:' in text and 'WHOLE TISSUE STATISTICS:' in text:
                inside_block = text.split('INSIDE ROI TISSUE STATISTICS:')[1].split('WHOLE TISSUE STATISTICS:')[0]
                whole_block = text.split('WHOLE TISSUE STATISTICS:')[1]
            else:
                inside_block = text
            parts = txt.parts
            batch = next((p for p in parts if p.lower().startswith('batch')), '')
            sex = 'Male' if any(p.lower()=='male' for p in parts) else ('Female' if any(p.lower()=='female' for p in parts) else '')
            row = {'batch': batch, 'sex': sex, 'filename': txt.name}
            for k,p in patterns.items():
                mi = p.search(inside_block)
                mw = p.search(whole_block)
                row[f'inside_{k}'] = mi.group(1) if mi else ''
                row[f'whole_{k}'] = mw.group(1) if mw else ''
            rows.append(row)
    return rows

try:
    import pandas as pd
    if DATA_CSV.exists():
        master = pd.read_csv(DATA_CSV)
    else:
        downloads_root = Path(r"c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA Scans")
        rows = scan_and_build(downloads_root)
        master = pd.DataFrame(rows)
        try:
            master.to_csv(DATA_CSV, index=False)
        except Exception:
            pass
    if not master.empty:
        df_batch = master[master['batch'].str.lower()==batch_name.lower()]
        if not df_batch.empty:
            display(df_batch.reset_index(drop=True))
        else:
            print('No week-1 rows for', batch_name)
    else:
        print('No week-1 files found anywhere')
except Exception as e:
    print('Error building/displaying week-1 table:', e)


,batch,sex,filename,path,inside_sample_area,whole_sample_area,inside_bone_area,whole_bone_area,inside_total_weight,whole_total_weight,...,inside_lean_weight,whole_lean_weight,inside_fat_weight,whole_fat_weight,inside_fat_percent,whole_fat_percent,inside_BMC,whole_BMC,inside_BMD,whole_BMD
0,Batch 3,Female,B3_F_0.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,26.997,30.463,8.310,10.783,23.8872,27.5536,...,17.2997,19.2610,6.0269,7.3139,25.837,27.522,0.56056,0.97869,67.455,90.764
1,Batch 3,Female,B3_F_1.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,26.630,29.246,8.415,10.197,22.6607,25.4537,...,16.1375,17.3487,5.9351,7.1635,26.889,29.224,0.58814,0.94155,69.894,92.338
2,Batch 3,Female,B3_F_2.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,29.621,33.028,8.686,10.894,26.1507,29.8926,...,16.5273,18.4209,9.0351,10.4949,35.345,36.295,0.58821,0.97677,67.718,89.665
3,Batch 3,Female,B3_F_3.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,27.465,30.423,8.499,10.404,23.8801,27.0968,...,16.4345,17.8458,6.8729,8.3068,29.488,31.763,0.57269,0.94423,67.384,90.758
4,Batch 3,Female,B3_F_4.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,28.403,31.685,8.218,10.441,26.1111,30.0057,...,19.6542,21.7387,5.9148,7.3196,23.133,25.189,0.54210,0.94735,65.964,90.736
5,Batch 3,Male,B3_M_0.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,31.648,35.271,9.335,11.981,32.8641,36.9747,...,20.0577,21.4947,12.1344,14.3344,37.694,40.008,0.67199,1.14558,71.989,95.620
6,Batch 3,Male,B3_M_1.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,31.645,36.147,10.072,12.810,33.0451,37.6966,...,20.2612,21.4977,12.0280,14.9589,37.251,41.032,0.75590,1.24004,75.046,96.800
7,Batch 3,Male,B3_M_2.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,29.928,33.780,8.607,11.313,30.9445,35.1592,...,20.7126,23.1708,9.6390,10.9761,31.758,32.144,0.59282,1.01230,68.873,89.479
8,Batch 3,Male,B3_M_3.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,27.493,31.612,8.043,10.704,25.4151,29.5285,...,18.6875,21.3781,6.1703,7.1795,24.822,25.140,0.55732,0.97096,69.297,90.707
9,Batch 3,Male,B3_M_4.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,30.373,34.760,8.598,11.275,31.5427,36.2000,...,22.8174,25.6769,8.1231,9.4873,26.254,26.980,0.60216,1.03583,70.030,91.870


## Batch 3 — 2-week post-treatment summary



In [1]:
# Batch 3 — 2-week post-treatment summary
from pathlib import Path
import re

DATA_CSV2 = Path(r'../data/week2_reports.csv').resolve()
batch_name = 'Batch 3'

def scan_and_build_week(downloads_root, week_num):
    week_re = re.compile(rf'(?:week[\s_-]*{week_num}|{week_num}[\s_-]*week|wk[\s_-]*{week_num}|week{week_num}|{week_num}week)', re.I)
    patterns = {
        'sample_area': re.compile(r"Sample Area:\s*([0-9.]+)\s*cm\^2"),
        'bone_area': re.compile(r"Bone Area:\s*([0-9.]+)\s*cm\^2"),
        'total_weight': re.compile(r"Total Weight:\s*([0-9.]+)\s*g"),
        'soft_weight': re.compile(r"Soft Weight:\s*([0-9.]+)\s*g"),
        'lean_weight': re.compile(r"Lean Weight:\s*([0-9.]+)\s*g"),
        'fat_weight': re.compile(r"Fat Weight:\s*([0-9.]+)\s*g"),
        'fat_percent': re.compile(r"Fat Percent:\s*([0-9.]+)"),
        'BMC': re.compile(r"BMC:\s*([0-9.]+)\s*g"),
        'BMD': re.compile(r"BMD:\s*([0-9.]+)\s*mg/cm\^2"),
    }
    rows = []
    for txt in downloads_root.rglob('*.txt'):
        nl = str(txt).lower()
        if week_re.search(nl):
            text = txt.read_text(encoding='utf-8', errors='replace')
            inside_block = ''
            whole_block = ''
            if 'INSIDE ROI TISSUE STATISTICS:' in text and 'WHOLE TISSUE STATISTICS:' in text:
                inside_block = text.split('INSIDE ROI TISSUE STATISTICS:')[1].split('WHOLE TISSUE STATISTICS:')[0]
                whole_block = text.split('WHOLE TISSUE STATISTICS:')[1]
            else:
                inside_block = text
            parts = txt.parts
            batch = next((p for p in parts if p.lower().startswith('batch')), '')
            sex = 'Male' if any(p.lower()=='male' for p in parts) else ('Female' if any(p.lower()=='female' for p in parts) else '')
            row = {'batch': batch, 'sex': sex, 'filename': txt.name}
            for k,p in patterns.items():
                mi = p.search(inside_block)
                mw = p.search(whole_block)
                row[f'inside_{k}'] = mi.group(1) if mi else ''
                row[f'whole_{k}'] = mw.group(1) if mw else ''
            rows.append(row)
    return rows

try:
    import pandas as pd
    if DATA_CSV2.exists():
        master2 = pd.read_csv(DATA_CSV2)
    else:
        downloads_root = Path(r"c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA Scans")
        rows2 = scan_and_build_week(downloads_root, 2)
        master2 = pd.DataFrame(rows2)
        try:
            master2.to_csv(DATA_CSV2, index=False)
        except Exception:
            pass
    if not master2.empty:
        df_batch2 = master2[master2['batch'].str.lower()==batch_name.lower()]
        if not df_batch2.empty:
            display(df_batch2.reset_index(drop=True))
        else:
            print('No week-2 rows for', batch_name)
    else:
        print('No week-2 files found anywhere')
except Exception as e:
    print('Error building/displaying week-2 table:', e)


,batch,sex,filename,inside_sample_area,whole_sample_area,inside_bone_area,whole_bone_area,inside_total_weight,whole_total_weight,inside_soft_weight,...,inside_lean_weight,whole_lean_weight,inside_fat_weight,whole_fat_weight,inside_fat_percent,whole_fat_percent,inside_BMC,whole_BMC,inside_BMD,whole_BMD
0,Batch 3,Female,B3_F_0.txt,27.755,30.286,9.086,10.623,24.9872,27.6699,24.2817,...,16.3709,17.4370,7.9109,9.2781,32.580,34.730,0.70549,0.95488,77.643,89.885
1,Batch 3,Female,B3_F_1.txt,26.866,29.512,7.865,9.198,22.7166,25.6707,22.1458,...,16.7825,18.6958,5.3632,6.1894,24.218,24.872,0.57085,0.78550,72.578,85.399
2,Batch 3,Female,B3_F_2.txt,28.506,30.991,8.639,9.976,26.1030,28.8104,25.5267,...,16.9225,18.2914,8.6041,9.7507,33.707,34.772,0.57629,0.76824,66.706,77.011
3,Batch 3,Female,B3_F_3.txt,27.392,30.152,8.565,10.157,24.4548,27.5562,23.8291,...,17.1305,18.6460,6.6987,8.0342,28.111,30.113,0.62571,0.87597,73.057,86.247
4,Batch 3,Female,B3_F_4.txt,28.897,31.754,8.596,10.200,26.6620,29.9340,26.0474,...,19.7920,21.6999,6.2555,7.3708,24.016,25.355,0.61454,0.86340,71.490,84.648
5,Batch 3,Male,B3_M_0.txt,32.888,36.069,10.285,11.814,32.2950,35.4958,31.5455,...,19.7859,21.1516,11.7596,13.3594,37.278,38.711,0.74945,0.98484,72.865,83.364
6,Batch 3,Male,B3_M_1.txt,32.864,36.483,9.110,10.549,31.1102,34.6191,30.4152,...,21.9717,24.0272,8.4435,9.6801,27.761,28.718,0.69495,0.91179,76.283,86.432
7,Batch 3,Male,B3_M_2.txt,29.860,32.674,8.711,10.234,29.2176,32.4387,28.6312,...,19.7775,21.5238,8.8537,10.1237,30.923,31.989,0.58646,0.79117,67.324,77.310
8,Batch 3,Male,B3_M_3.txt,26.503,29.608,7.973,9.622,23.8102,27.3267,23.2774,...,17.9951,20.3380,5.2823,6.2402,22.693,23.479,0.53283,0.74843,66.828,77.786
9,Batch 3,Male,B3_M_4 Attempt 2.txt,30.048,33.434,8.348,10.142,29.6689,33.4780,29.0678,...,23.2667,25.7055,5.8012,6.9200,19.957,21.210,0.60109,0.85256,72.002,84.064


## Batch 3 — 3-week post-treatment summary



In [1]:
# Batch 3 — 3-week post-treatment summary
from pathlib import Path
import re

DATA_CSV3 = Path(r'../data/week3_reports.csv').resolve()
batch_name = 'Batch 3'

try:
    import pandas as pd
    if DATA_CSV3.exists():
        master3 = pd.read_csv(DATA_CSV3)
    else:
        downloads_root = Path(r"c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA Scans")
        rows3 = scan_and_build_week(downloads_root, 3)
        master3 = pd.DataFrame(rows3)
        try:
            master3.to_csv(DATA_CSV3, index=False)
        except Exception:
            pass
    if not master3.empty:
        df_batch3 = master3[master3['batch'].str.lower()==batch_name.lower()]
        if not df_batch3.empty:
            display(df_batch3.reset_index(drop=True))
        else:
            print('No week-3 rows for', batch_name)
    else:
        print('No week-3 files found anywhere')
except Exception as e:
    print('Error building/displaying week-3 table:', e)


,batch,sex,filename,inside_sample_area,whole_sample_area,inside_bone_area,whole_bone_area,inside_total_weight,whole_total_weight,inside_soft_weight,...,inside_lean_weight,whole_lean_weight,inside_fat_weight,whole_fat_weight,inside_fat_percent,whole_fat_percent,inside_BMC,whole_BMC,inside_BMD,whole_BMD
0,Batch 3,Female,B3_F_0.txt,25.935,27.830,9.112,10.097,23.9827,25.9556,23.2816,...,16.6372,17.6628,6.6444,7.4396,28.539,29.637,0.70110,0.85313,76.944,84.490
1,Batch 3,Female,B3_F_1.txt,26.191,28.564,8.357,9.633,21.8805,24.3701,21.2729,...,15.9136,17.0106,5.3593,6.5599,25.193,27.831,0.60754,0.79964,72.698,83.008
2,Batch 3,Female,B3_F_2.txt,28.427,30.124,8.462,9.184,25.5606,27.2030,24.9803,...,16.4215,17.2061,8.5588,9.3382,34.262,35.180,0.58028,0.65873,68.577,71.727
3,Batch 3,Female,B3_F_3.txt,26.259,28.733,8.959,10.368,24.6329,27.4751,23.9286,...,16.8710,18.1069,7.0576,8.4197,29.494,31.741,0.70429,0.94851,78.613,91.488
4,Batch 3,Male,B3_M_0.txt,29.364,32.273,9.380,10.516,29.5048,32.5606,28.7680,...,19.4795,21.0179,9.2886,10.6424,32.288,33.614,0.73672,0.90029,78.545,85.610
5,Batch 3,Male,B3_M_1.txt,31.131,34.236,8.784,10.290,29.6032,32.9519,28.9617,...,22.1145,24.1944,6.8473,7.9077,23.642,24.633,0.64141,0.84979,73.021,82.587
6,Batch 3,Male,B3_M_2.txt,29.584,33.073,8.755,10.355,29.0756,32.8726,28.4877,...,19.3251,21.5704,9.1627,10.5217,32.164,32.786,0.58785,0.78048,67.147,75.371
7,Batch 3,Male,B3_M_3.txt,24.316,27.399,7.109,8.420,20.9640,24.0295,20.4640,...,15.0669,17.0652,5.3971,6.2838,26.374,26.912,0.50003,0.68055,70.341,80.827
8,Batch 3,Male,B3_M_4.txt,29.917,32.668,8.573,9.869,29.4143,32.4288,28.7765,...,21.7282,23.1610,7.0483,8.4308,24.493,26.687,0.63775,0.83699,74.386,84.812
